# AstroCLIMB 02 — multimodal pair generation and validation

This notebook turns the compact metadata index created by notebook 01 into balanced, DOI-disjoint training and validation manifests. It generates all ten valid class–modality cells without loading the 72 GB image column.

Before running on Kaggle, attach the private output Dataset from notebook 01 containing `metadata_index.pkl`. Internet and a GPU are not required for this notebook. The generated manifests deliberately contain metadata for auditing and later object resolution; DOI, UUID, and row identifiers must not be used as model features.

In [1]:
from pathlib import Path
import collections
import csv
import hashlib
import json
import math
import pickle
import random
import re
import unicodedata

SEED = 2026
VALIDATION_FRACTION = 0.20
TRAIN_PAIRS_PER_CELL = 16_000
VALIDATION_PAIRS_PER_CELL = 4_000
HARD_NEGATIVE_FRACTION = 0.50
MAX_SAME_PAPER_PAIRS_PER_DOI = 60
MAX_RELATED_PAIRS_PER_EDGE = 40
MAX_UNRELATED_PAIRS_PER_DOI_PAIR = 12
SMOKE_TEST = False  # True creates 100 train and 40 validation pairs per valid cell

# Set this explicitly only if automatic discovery selects the wrong file.
METADATA_INDEX_PATH = None
OUTPUT_DIR = Path('/kaggle/working/astroclimb_02') if Path('/kaggle/working').exists() else Path('results/astroclimb_02')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    TRAIN_PAIRS_PER_CELL = 100
    VALIDATION_PAIRS_PER_CELL = 40

LABELS = ('same_figure', 'same_paper', 'related_papers', 'unrelated_papers')
MODALITIES = ('text-text', 'text-image', 'image-image')
VALID_CELLS = tuple(
    (label, modality)
    for label in LABELS
    for modality in MODALITIES
    if label != 'same_figure' or modality == 'text-image'
)
print('Output:', OUTPUT_DIR.resolve())
print('Valid cells:', len(VALID_CELLS), VALID_CELLS)

Output: /kaggle/working/astroclimb_02
Valid cells: 10 (('same_figure', 'text-image'), ('same_paper', 'text-text'), ('same_paper', 'text-image'), ('same_paper', 'image-image'), ('related_papers', 'text-text'), ('related_papers', 'text-image'), ('related_papers', 'image-image'), ('unrelated_papers', 'text-text'), ('unrelated_papers', 'text-image'), ('unrelated_papers', 'image-image'))


## Locate and load the compact metadata index

The loader accepts the dictionary-format index written by notebook 01 and rejects older class-instance caches. Candidate files are ranked by whether their path contains `astroclimb_01`.

In [2]:
def candidate_index_paths():
    roots = [Path('/kaggle/input'), Path('.'), Path('notebooks')]
    found = []
    for root in roots:
        if root.exists():
            found.extend(root.rglob('metadata_index.pkl'))
    return sorted(set(found), key=lambda p: ('astroclimb_01' not in str(p).lower(), len(str(p))))

def load_compact_index(explicit_path=None):
    paths = [Path(explicit_path)] if explicit_path else candidate_index_paths()
    failures = []
    for path in paths:
        try:
            with path.open('rb') as handle:
                value = pickle.load(handle)
            if isinstance(value, dict) and isinstance(value.get('records'), list):
                required = {'meta_row', 'uuid', 'image_id', 'doi', 'caption', 'references', 'citations'}
                if value['records'] and required.issubset(value['records'][0]):
                    return path, value
            failures.append(f'{path}: incompatible format')
        except Exception as exc:
            failures.append(f'{path}: {type(exc).__name__}: {exc}')
    message = '\n'.join(failures[:10]) or 'No metadata_index.pkl files were found.'
    raise FileNotFoundError(
        'Attach the astroclimb_01 output Dataset or set METADATA_INDEX_PATH.\n' + message
    )

INDEX_PATH, metadata_index = load_compact_index(METADATA_INDEX_PATH)
records = metadata_index['records']
print('Loaded:', INDEX_PATH)
print('Metadata records:', f'{len(records):,}')
print('HF source:', metadata_index.get('hf_dataset'), metadata_index.get('hf_split'))
print('Image scan complete:', metadata_index.get('image_scan_complete'))
assert len(records) > 10_000, 'The attached index looks like a partial smoke-test artifact.'

Loaded: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-01/astroclimb_01/metadata_index.pkl
Metadata records: 94,233
HF source: adsabs/AstroCLIMB train
Image scan complete: True


## Normalize records and construct the paper graph

All relationship checks use the complete citation graph. Empty-DOI or empty-caption records are excluded because they cannot support every requested modality.

In [3]:
TOKEN_RE = re.compile(r'[a-z0-9][a-z0-9+_.-]{2,}')
STOPWORDS = frozenset({
    'about','after','also','are','between','bottom','caption','color','data','different',
    'figure','figures','from','left','lines','model','observed','our','panel','panels',
    'paper','results','right','same','shown','shows','the','their','this','top','using',
    'values','where','which','with'
})

def norm_doi(value):
    doi = str(value or '').strip().casefold()
    doi = re.sub(r'^https?://(?:dx\.)?doi\.org/', '', doi)
    return doi.rstrip('.,; ')

def norm_set(value):
    return frozenset(filter(None, (norm_doi(x) for x in (value or ()))))

def tokens(text):
    text = unicodedata.normalize('NFKC', str(text or '')).casefold()
    return frozenset(t for t in TOKEN_RE.findall(text) if t not in STOPWORDS and not t.isdigit())

clean = []
for source in records:
    doi = norm_doi(source.get('doi'))
    caption = str(source.get('caption') or '').strip()
    if not doi or not caption:
        continue
    clean.append({
        'row': int(source['meta_row']),
        'uuid': str(source.get('uuid') or ''),
        'image_id': str(source.get('image_id') or ''),
        'doi': doi,
        'caption': caption,
        'references': norm_set(source.get('references')),
        'citations': norm_set(source.get('citations')),
    })

by_row = {r['row']: r for r in clean}
by_doi = collections.defaultdict(list)
for record in clean:
    by_doi[record['doi']].append(record)
known_dois = set(by_doi)
adjacency = {doi: set() for doi in known_dois}
for record in clean:
    for neighbor in record['references'] | record['citations']:
        if neighbor in known_dois and neighbor != record['doi']:
            adjacency[record['doi']].add(neighbor)
            adjacency[neighbor].add(record['doi'])
edges = {tuple(sorted((left, right))) for left, neighbors in adjacency.items() for right in neighbors}
print('Eligible records:', f'{len(clean):,}')
print('Paper DOIs:', f'{len(by_doi):,}')
print('Undirected citation edges:', f'{len(edges):,}')
print('Papers with >=2 figures:', f'{sum(len(v) >= 2 for v in by_doi.values()):,}')

Eligible records: 94,233
Paper DOIs: 10,000
Undirected citation edges: 27,003
Papers with >=2 figures: 9,385


## Create a DOI-disjoint split

Papers are split before pairs are sampled. Related-paper examples use citation edges whose two endpoints are inside the same split.

In [4]:
rng = random.Random(SEED)
dois = sorted(by_doi)
rng.shuffle(dois)
n_validation = max(1, round(len(dois) * VALIDATION_FRACTION))
validation_dois = set(dois[:n_validation])
train_dois = set(dois[n_validation:])
assert train_dois.isdisjoint(validation_dois)

split_records = {
    'train': [r for r in clean if r['doi'] in train_dois],
    'validation': [r for r in clean if r['doi'] in validation_dois],
}
split_dois = {'train': train_dois, 'validation': validation_dois}
split_edges = {
    name: sorted(edge for edge in edges if edge[0] in doi_set and edge[1] in doi_set)
    for name, doi_set in split_dois.items()
}
for name in ('train', 'validation'):
    print(name, {
        'records': len(split_records[name]),
        'papers': len(split_dois[name]),
        'internal_edges': len(split_edges[name]),
        'multi_figure_papers': sum(len(by_doi[d]) >= 2 for d in split_dois[name]),
    })
    assert split_edges[name], f'{name} contains no internal citation edges; change SEED.'

train {'records': 75120, 'papers': 8000, 'internal_edges': 17239, 'multi_figure_papers': 7488}
validation {'records': 19113, 'papers': 2000, 'internal_edges': 1107, 'multi_figure_papers': 1897}


## Balanced multimodal sampler

Object-pair identity includes both the source metadata row and object modality. The sampler treats input order as symmetric, prevents duplicates, caps repeated paper/edge contributions, and mixes lexical hard negatives with random non-edges.

In [5]:
def object_types(modality, rng):
    if modality == 'text-text':
        return 'text', 'text'
    if modality == 'image-image':
        return 'image', 'image'
    return ('text', 'image') if rng.random() < 0.5 else ('image', 'text')

def pair_key(row_a, type_a, row_b, type_b):
    return tuple(sorted(((int(row_a), type_a), (int(row_b), type_b))))

def make_pair(a, b, modality, relationship, sampling, rng, shared_token=''):
    type_a, type_b = object_types(modality, rng)
    return {
        'obj_1_row': a['row'], 'obj_1_type': type_a,
        'obj_2_row': b['row'], 'obj_2_type': type_b,
        'modality': modality, 'relationship': relationship,
        'sampling': sampling, 'shared_token': shared_token,
    }

def add_unique(selected, pair):
    key = pair_key(pair['obj_1_row'], pair['obj_1_type'], pair['obj_2_row'], pair['obj_2_type'])
    if key in selected:
        return False
    selected[key] = pair
    return True

def sample_same_figure(rows, count, modality, rng):
    if modality != 'text-image':
        raise ValueError('same_figure is valid only for text-image')
    if count > len(rows):
        raise ValueError(f'Requested {count} same-figure pairs from {len(rows)} records')
    return [make_pair(r, r, modality, 'same_figure', 'exact_alignment', rng) for r in rng.sample(rows, count)]

def sample_same_paper(doi_pool, count, modality, rng):
    eligible = [d for d in doi_pool if len(by_doi[d]) >= 2]
    rng.shuffle(eligible)
    selected, per_doi = {}, collections.Counter()
    attempts, max_attempts = 0, max(50_000, count * 300)
    while len(selected) < count and attempts < max_attempts:
        doi = eligible[attempts % len(eligible)] if attempts < len(eligible) * 2 else rng.choice(eligible)
        attempts += 1
        if per_doi[doi] >= MAX_SAME_PAPER_PAIRS_PER_DOI:
            continue
        a, b = rng.sample(by_doi[doi], 2)
        pair = make_pair(a, b, modality, 'same_paper', 'same_doi', rng)
        if add_unique(selected, pair):
            per_doi[doi] += 1
    if len(selected) != count:
        raise RuntimeError(f'same_paper/{modality}: sampled {len(selected)}/{count}; raise the cap or lower the count')
    return list(selected.values())

def sample_related(edge_pool, count, modality, rng):
    edge_pool = list(edge_pool)
    rng.shuffle(edge_pool)
    selected, per_edge = {}, collections.Counter()
    attempts, max_attempts = 0, max(50_000, count * 300)
    while len(selected) < count and attempts < max_attempts:
        edge = edge_pool[attempts % len(edge_pool)] if attempts < len(edge_pool) * 2 else rng.choice(edge_pool)
        attempts += 1
        if per_edge[edge] >= MAX_RELATED_PAIRS_PER_EDGE:
            continue
        left, right = edge
        if rng.random() < 0.5:
            left, right = right, left
        a, b = rng.choice(by_doi[left]), rng.choice(by_doi[right])
        pair = make_pair(a, b, modality, 'related_papers', 'citation_edge', rng)
        if add_unique(selected, pair):
            per_edge[edge] += 1
    if len(selected) != count:
        raise RuntimeError(f'related/{modality}: sampled {len(selected)}/{count}; raise the cap or lower the count')
    return list(selected.values())

def build_token_index(rows):
    tokens_by_row = {r['row']: tokens(r['caption']) for r in rows}
    token_rows = collections.defaultdict(list)
    for record in rows:
        for token in tokens_by_row[record['row']]:
            token_rows[token].append(record)
    upper = max(100, len(rows) // 100)
    useful = {t for t, candidates in token_rows.items() if 2 <= len(candidates) <= upper}
    return tokens_by_row, token_rows, useful

def sample_unrelated(rows, count, modality, hard_fraction, rng):
    selected, per_pair = {}, collections.Counter()
    tokens_by_row, token_rows, useful = build_token_index(rows)
    hard_target = round(count * hard_fraction)

    def unrelated(a, b):
        return a['doi'] != b['doi'] and b['doi'] not in adjacency[a['doi']]

    attempts, max_attempts = 0, max(100_000, hard_target * 500)
    while len(selected) < hard_target and attempts < max_attempts:
        attempts += 1
        a = rng.choice(rows)
        choices = tuple(tokens_by_row[a['row']] & useful)
        if not choices:
            continue
        token = rng.choice(choices)
        b = rng.choice(token_rows[token])
        doi_pair = tuple(sorted((a['doi'], b['doi'])))
        if not unrelated(a, b) or per_pair[doi_pair] >= MAX_UNRELATED_PAIRS_PER_DOI_PAIR:
            continue
        pair = make_pair(a, b, modality, 'unrelated_papers', 'shared_caption_token', rng, token)
        if add_unique(selected, pair):
            per_pair[doi_pair] += 1

    attempts, max_attempts = 0, max(100_000, count * 300)
    while len(selected) < count and attempts < max_attempts:
        attempts += 1
        a, b = rng.choice(rows), rng.choice(rows)
        doi_pair = tuple(sorted((a['doi'], b['doi'])))
        if not unrelated(a, b) or per_pair[doi_pair] >= MAX_UNRELATED_PAIRS_PER_DOI_PAIR:
            continue
        pair = make_pair(a, b, modality, 'unrelated_papers', 'random_nonedge', rng)
        if add_unique(selected, pair):
            per_pair[doi_pair] += 1
    if len(selected) != count:
        raise RuntimeError(f'unrelated/{modality}: sampled {len(selected)}/{count}; raise the cap or lower the count')
    return list(selected.values())

def generate_split(name, count_per_cell, seed):
    local_rng = random.Random(seed)
    rows, doi_pool, edge_pool = split_records[name], split_dois[name], split_edges[name]
    output = []
    for label, modality in VALID_CELLS:
        print(f'{name}: {label}/{modality} ...', end=' ')
        if label == 'same_figure':
            part = sample_same_figure(rows, count_per_cell, modality, local_rng)
        elif label == 'same_paper':
            part = sample_same_paper(doi_pool, count_per_cell, modality, local_rng)
        elif label == 'related_papers':
            part = sample_related(edge_pool, count_per_cell, modality, local_rng)
        else:
            part = sample_unrelated(rows, count_per_cell, modality, HARD_NEGATIVE_FRACTION, local_rng)
        output.extend(part)
        print(len(part))
    local_rng.shuffle(output)
    return output

## Generate manifests

In [6]:
train_pairs = generate_split('train', TRAIN_PAIRS_PER_CELL, SEED + 1)
validation_pairs = generate_split('validation', VALIDATION_PAIRS_PER_CELL, SEED + 2)
print('Generated:', {'train': len(train_pairs), 'validation': len(validation_pairs)})

train: same_figure/text-image ... 16000
train: same_paper/text-text ... 16000
train: same_paper/text-image ... 16000
train: same_paper/image-image ... 16000
train: related_papers/text-text ... 16000
train: related_papers/text-image ... 16000
train: related_papers/image-image ... 16000
train: unrelated_papers/text-text ... 16000
train: unrelated_papers/text-image ... 16000
train: unrelated_papers/image-image ... 16000
validation: same_figure/text-image ... 4000
validation: same_paper/text-text ... 4000
validation: same_paper/text-image ... 4000
validation: same_paper/image-image ... 4000
validation: related_papers/text-text ... 4000
validation: related_papers/text-image ... 4000
validation: related_papers/image-image ... 4000
validation: unrelated_papers/text-text ... 4000
validation: unrelated_papers/text-image ... 4000
validation: unrelated_papers/image-image ... 4000
Generated: {'train': 160000, 'validation': 40000}


## Validate relationships, balance, uniqueness, and leakage

This is a hard gate: the notebook will stop instead of saving manifests if any relationship is incorrect, a pair is duplicated, a class-modality count is wrong, or a DOI crosses the split.

In [7]:
def validate_split(name, pairs, expected_per_cell):
    counts = collections.Counter((p['relationship'], p['modality']) for p in pairs)
    assert set(counts) == set(VALID_CELLS), (set(counts), set(VALID_CELLS))
    assert all(counts[cell] == expected_per_cell for cell in VALID_CELLS), counts
    keys = [pair_key(p['obj_1_row'], p['obj_1_type'], p['obj_2_row'], p['obj_2_type']) for p in pairs]
    assert len(keys) == len(set(keys)), f'{name} contains duplicate symmetric object pairs'
    allowed = split_dois[name]
    for p in pairs:
        a, b = by_row[p['obj_1_row']], by_row[p['obj_2_row']]
        assert a['doi'] in allowed and b['doi'] in allowed
        if p['relationship'] == 'same_figure':
            assert a['row'] == b['row'] and p['modality'] == 'text-image'
        elif p['relationship'] == 'same_paper':
            assert a['doi'] == b['doi'] and a['row'] != b['row']
        elif p['relationship'] == 'related_papers':
            assert a['doi'] != b['doi'] and b['doi'] in adjacency[a['doi']]
        else:
            assert a['doi'] != b['doi'] and b['doi'] not in adjacency[a['doi']]
    return counts

train_counts = validate_split('train', train_pairs, TRAIN_PAIRS_PER_CELL)
validation_counts = validate_split('validation', validation_pairs, VALIDATION_PAIRS_PER_CELL)
train_rows_used = {p[k] for p in train_pairs for k in ('obj_1_row', 'obj_2_row')}
validation_rows_used = {p[k] for p in validation_pairs for k in ('obj_1_row', 'obj_2_row')}
assert train_rows_used.isdisjoint(validation_rows_used)
assert train_dois.isdisjoint(validation_dois)
print('Validation passed.')
print('Train rows used:', f'{len(train_rows_used):,}')
print('Validation rows used:', f'{len(validation_rows_used):,}')
print('Train sampling:', collections.Counter(p['sampling'] for p in train_pairs))
print('Validation sampling:', collections.Counter(p['sampling'] for p in validation_pairs))

Validation passed.
Train rows used: 72,196
Validation rows used: 18,254
Train sampling: Counter({'same_doi': 48000, 'citation_edge': 48000, 'shared_caption_token': 24000, 'random_nonedge': 24000, 'exact_alignment': 16000})
Validation sampling: Counter({'same_doi': 12000, 'citation_edge': 12000, 'random_nonedge': 6000, 'shared_caption_token': 6000, 'exact_alignment': 4000})


## Save compact, reproducible artifacts

The manifests contain no image bytes or full captions. `obj_1_row` and `obj_2_row` refer to `meta_row` in the notebook 01 index and to the stable row order of the Hugging Face split.

In [8]:
MANIFEST_FIELDS = (
    'pair_id','split','obj_1_row','obj_1_type','obj_1_uuid','obj_1_image_id','obj_1_doi',
    'obj_2_row','obj_2_type','obj_2_uuid','obj_2_image_id','obj_2_doi',
    'modality','relationship','sampling','shared_token'
)

def write_manifest(path, split, pairs):
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        for index, pair in enumerate(pairs):
            a, b = by_row[pair['obj_1_row']], by_row[pair['obj_2_row']]
            writer.writerow({
                'pair_id': f'{split}_{index:08d}', 'split': split,
                'obj_1_row': a['row'], 'obj_1_type': pair['obj_1_type'],
                'obj_1_uuid': a['uuid'], 'obj_1_image_id': a['image_id'], 'obj_1_doi': a['doi'],
                'obj_2_row': b['row'], 'obj_2_type': pair['obj_2_type'],
                'obj_2_uuid': b['uuid'], 'obj_2_image_id': b['image_id'], 'obj_2_doi': b['doi'],
                'modality': pair['modality'], 'relationship': pair['relationship'],
                'sampling': pair['sampling'], 'shared_token': pair['shared_token'],
            })

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

train_path = OUTPUT_DIR / 'hf_train_multimodal_pairs.csv'
validation_path = OUTPUT_DIR / 'hf_validation_multimodal_pairs.csv'
doi_path = OUTPUT_DIR / 'hf_doi_split.csv'
write_manifest(train_path, 'train', train_pairs)
write_manifest(validation_path, 'validation', validation_pairs)
with doi_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(('paper_doi', 'split'))
    writer.writerows((doi, 'train' if doi in train_dois else 'validation') for doi in sorted(known_dois))

summary = {
    'configuration': {
        'seed': SEED, 'validation_fraction': VALIDATION_FRACTION,
        'train_pairs_per_cell': TRAIN_PAIRS_PER_CELL,
        'validation_pairs_per_cell': VALIDATION_PAIRS_PER_CELL,
        'hard_negative_fraction': HARD_NEGATIVE_FRACTION,
        'max_same_paper_pairs_per_doi': MAX_SAME_PAPER_PAIRS_PER_DOI,
        'max_related_pairs_per_edge': MAX_RELATED_PAIRS_PER_EDGE,
        'max_unrelated_pairs_per_doi_pair': MAX_UNRELATED_PAIRS_PER_DOI_PAIR,
        'smoke_test': SMOKE_TEST,
    },
    'source': {
        'metadata_index_path': str(INDEX_PATH),
        'hf_dataset': metadata_index.get('hf_dataset'),
        'hf_split': metadata_index.get('hf_split'),
        'eligible_records': len(clean), 'paper_dois': len(known_dois), 'citation_edges': len(edges),
    },
    'train': {
        'rows': len(train_pairs), 'papers': len(train_dois), 'records': len(split_records['train']),
        'internal_edges': len(split_edges['train']), 'unique_source_rows_used': len(train_rows_used),
        'cell_counts': {f'{a}|{b}': train_counts[(a,b)] for a,b in VALID_CELLS},
        'sampling_counts': dict(collections.Counter(p['sampling'] for p in train_pairs)),
    },
    'validation': {
        'rows': len(validation_pairs), 'papers': len(validation_dois), 'records': len(split_records['validation']),
        'internal_edges': len(split_edges['validation']), 'unique_source_rows_used': len(validation_rows_used),
        'cell_counts': {f'{a}|{b}': validation_counts[(a,b)] for a,b in VALID_CELLS},
        'sampling_counts': dict(collections.Counter(p['sampling'] for p in validation_pairs)),
    },
}
summary_path = OUTPUT_DIR / 'pair_generation_summary.json'
with summary_path.open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2, sort_keys=True)
checksums = {p.name: sha256_file(p) for p in (train_path, validation_path, doi_path, summary_path)}
with (OUTPUT_DIR / 'sha256_checksums.json').open('w', encoding='utf-8') as handle:
    json.dump(checksums, handle, indent=2, sort_keys=True)
print(json.dumps(summary, indent=2, sort_keys=True))
print('Artifacts:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path.name}: {path.stat().st_size / 1024**2:.2f} MiB')

{
  "configuration": {
    "hard_negative_fraction": 0.5,
    "max_related_pairs_per_edge": 40,
    "max_same_paper_pairs_per_doi": 60,
    "max_unrelated_pairs_per_doi_pair": 12,
    "seed": 2026,
    "smoke_test": false,
    "train_pairs_per_cell": 16000,
    "validation_fraction": 0.2,
    "validation_pairs_per_cell": 4000
  },
  "source": {
    "citation_edges": 27003,
    "eligible_records": 94233,
    "hf_dataset": "adsabs/AstroCLIMB",
    "hf_split": "train",
    "metadata_index_path": "/kaggle/input/datasets/syedmohaiminulhoque/astroclimb-01/astroclimb_01/metadata_index.pkl",
    "paper_dois": 10000
  },
  "train": {
    "cell_counts": {
      "related_papers|image-image": 16000,
      "related_papers|text-image": 16000,
      "related_papers|text-text": 16000,
      "same_figure|text-image": 16000,
      "same_paper|image-image": 16000,
      "same_paper|text-image": 16000,
      "same_paper|text-text": 16000,
      "unrelated_papers|image-image": 16000,
      "unrelated_paper

## Decision gate and handoff

A successful full run should contain exactly 160,000 training pairs and 40,000 validation pairs, with the expected count in every valid cell and no assertion failures. Save the entire `astroclimb_02` directory as a private Kaggle Dataset.

Send back `pair_generation_summary.json` and the final printed output. The following notebook will resolve the referenced rows, cache frozen embeddings, and run experiments E03–E06. Do not use the manifest DOI, UUID, row, or sampling columns as model inputs.